In [1]:
# import libreries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

In [2]:
# create connection with MySQL server
load_dotenv()

user = os.getenv('MYSQL_USER')
password = os.getenv('MYSQL_PASSWORD')
host = os.getenv('DWH_HOST')
port = os.getenv('DWH_PORT')

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/retail_dwh")

with engine.connect() as conn:
    result = conn.execute(text("SELECT 'Connection OK' AS status, DATABASE() AS current_db, VERSION() AS mysql_version"))
    row = result.fetchone()
    print(f"Status:        {row[0]}")
    print(f"Database:      {row[1]}")
    print(f"MySQL version: {row[2]}")

Status:        Connection OK
Database:      retail_dwh
MySQL version: 8.0.46


In [ ]:
# Show all the tables in the server
with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES;"))

    for row in result:
        print(row)

('dim_customer',)
('dim_date',)
('dim_geography',)
('dim_product',)
('fact_sales',)
('vw_daily_revenue',)
('vw_monthly_trend',)
('vw_return_rate',)
('vw_revenue_by_country',)
('vw_top_products',)


In [8]:
# Function for the queries
def consult(query):
    df = pd.read_sql(query, engine)
    return df

### Which products generate 80% of revenue?

In [ ]:
pareto_query = """
WITH revenue_per_product AS (
    SELECT
        p.product_key,
        p.stock_code,
        p.description,
        SUM(f.total_revenue) AS revenue
    FROM fact_sales f
    JOIN dim_product p ON f.product_key = p.product_key
    WHERE f.is_return = 0 AND p.is_internal = 0
    GROUP BY p.product_key, p.stock_code, p.description
)
, revenue_with_pct AS (
    SELECT
        stock_code,
        description,
        revenue,
        SUM(revenue) OVER () AS total_revenue,
        revenue / SUM(revenue) OVER () * 100 AS revenue_pct
    FROM revenue_per_product
)
SELECT
    stock_code,
    description,
    revenue,
    total_revenue,
    revenue_pct,
    SUM(revenue_pct) OVER (ORDER BY revenue DESC) AS cumulative_pct
FROM revenue_with_pct
ORDER BY revenue DESC
LIMIT 50;
"""
pareto = consult(pareto_query)

   stock_code                          description    revenue  total_revenue  \
0       22423             REGENCY CAKESTAND 3 TIER  174484.74    10272118.87   
1       23843          PAPER CRAFT , LITTLE BIRDIE  168469.60    10272118.87   
2      85123A   WHITE HANGING HEART T-LIGHT HOLDER  104518.80    10272118.87   
3       47566                        PARTY BUNTING   99504.33    10272118.87   
4      85099B              JUMBO BAG RED RETROSPOT   94340.05    10272118.87   
5       23166       MEDIUM CERAMIC TOP STORAGE JAR   81700.92    10272118.87   
6       23084                   RABBIT NIGHT LIGHT   66964.99    10272118.87   
7       22086      PAPER CHAIN KIT 50'S CHRISTMAS    64952.29    10272118.87   
8       84879        ASSORTED COLOUR BIRD ORNAMENT   59094.93    10272118.87   
9       79321                        CHILLI LIGHTS   54117.76    10272118.87   
10      22502       PICNIC BASKET WICKER 60 PIECES   51426.62    10272118.87   
11      22197                       POPC

In [13]:
pareto.head()

,stock_code,description,revenue,total_revenue,revenue_pct,cumulative_pct
0,22423,REGENCY CAKESTAND 3 TIER,174484.74,10272118.87,1.698625,1.698625
1,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,10272118.87,1.640067,3.338692
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104518.80,10272118.87,1.017500,4.356192
3,47566,PARTY BUNTING,99504.33,10272118.87,0.968684,5.324876
4,85099B,JUMBO BAG RED RETROSPOT,94340.05,10272118.87,0.918409,6.243285
